# Part C — Four-image stitching

Compare **SIFT, ORB and AKAZE** on the same four pagoda images, using image 3 as the reference. The detector comparison uses feather blending. The blending comparison applies feather and multi-band methods to the same selected homographies.

The notebook reads the saved local-refinement run. Its grid uses ratios 0.72/0.75/0.78 and thresholds 3/4/5/6 px; an expanded AKAZE edge search is stored separately. Set `rerun_pipeline = True` to recalculate the refinement grid. The [Part C analysis](../report_source/part_c.md) contains the recorded comparisons.


In [ ]:
import copy
import io
import json
import sys
from html import escape
from pathlib import Path

import numpy as np
from PIL import Image as PILImage
from IPython.display import HTML, Image, display

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src/part_c.py").is_file()
)
sys.path.insert(0, str(project_root / "src"))
import part_c

config = copy.deepcopy(part_c.DEFAULT_CONFIG)
config.update(
    profile="local_refinement",
    ratios=[0.72, 0.75, 0.78],
    thresholds_px=[3.0, 4.0, 5.0, 6.0],
)
output_root = project_root / part_c.DEFAULT_OUTPUT / "refinement"
rerun_pipeline = False
show_all_candidate_crops = False
run_directory = None  # Điền đường dẫn run nếu muốn xem lại một lần chạy cụ thể.

if rerun_pipeline:
    run_dir = part_c.run_experiments(
        project_root / part_c.DEFAULT_INPUT, output_root, config
    )
    selected_ratios = {"sift": 0.75, "orb": 0.78, "akaze": 0.78}
    choices = {}
    for method in config["methods"]:
        ratio = selected_ratios[method]
        choices[method] = {
            "case_id": part_c._case_id(method, ratio, 5.0),
            "reason": f"Reproduce the report's selected setting: Lowe ratio {ratio}, RANSAC threshold 5 px.",
        }
    part_c.finalize_selection(run_dir, choices)
    part_c.analyze_sensitivity(run_dir)
    part_c.compare_blending(run_dir)
elif run_directory is not None:
    run_dir = Path(run_directory)
    if not run_dir.is_absolute():
        run_dir = project_root / run_dir
else:
    run_dir = part_c.find_matching_run(
        project_root / part_c.DEFAULT_INPUT, output_root, config
    )

validation = part_c.validate_run(run_dir)
if not validation["valid"]:
    raise RuntimeError(validation["errors"])
manifest = json.loads((run_dir / "run.json").read_text())
config = copy.deepcopy(manifest["config"])
selection = json.loads((run_dir / "selection.json").read_text())
figures = json.loads((run_dir / "figures/figures.json").read_text())
print(
    "Run:", run_dir.relative_to(project_root) if run_dir.is_relative_to(project_root) else run_dir
)
print("Status:", manifest["status"], "| selection:", selection["status"])
print("OpenCV:", manifest["environment"]["opencv"])
print("Validation:", validation)


In [ ]:
def table(rows, columns):
    head = "".join(f"<th>{escape(key)}</th>" for key in columns)
    body = []
    for row in rows:
        cells = []
        for key in columns:
            value = row.get(key, "")
            if value not in (None, "") and key.endswith(
                ("_ms", "_px", "_pct", "_fraction", "from_feather")
            ):
                value = f"{float(value):.3f}"
            cells.append(f"<td>{escape(str(value))}</td>")
        cells = "".join(cells)
        body.append(f"<tr>{cells}</tr>")
    display(
        HTML(
            "<div style='overflow-x:auto'><table><thead><tr>"
            + head
            + "</tr></thead><tbody>"
            + "".join(body)
            + "</tbody></table></div>"
        )
    )


def show_image(path, max_width=2400):
    # Preview PNG không mất dữ liệu; ảnh 300 DPI gốc nằm trong thư mục run.
    with PILImage.open(path) as source:
        preview = source.convert("RGB")
        preview.thumbnail((max_width, max_width))
        buffer = io.BytesIO()
        preview.save(buffer, format="PNG")
    display(Image(data=buffer.getvalue()))


def show_figure(name):
    relative = figures["figures"].get(name)
    if relative is not None:
        show_image(run_dir / relative)


rows = part_c.read_csv(run_dir / "metrics.csv")
pairs = part_c.read_csv(run_dir / "pairs.csv")


## Input and metric conventions

Images retain their original resolutions and metadata order. The common reference is image 3; connections are **1→3, 2→3, 4→3**. Initial matches count KNN rows with two neighbours before the Lowe test. The inlier-ratio denominator is the number passing the Lowe test. RMSE is the forward pixel reprojection error on RANSAC inliers, aggregated by summed squared errors. Detected and used keypoints are counted separately.

All runs use one OpenCV CPU thread and OpenCL disabled. End-to-end timing includes grayscale conversion, detector creation, extraction, matching, homography, canvas, warp, blending and crop. It excludes I/O, metric calculation and figure generation.

In [ ]:
environment = manifest["environment"]
environment_rows = [{"component": "Python", "version": environment["python"].split()[0]}]
environment_rows.extend(
    {"component": name, "version": value} for name, value in environment["packages"].items()
)
table(environment_rows, ["component", "version"])

table(manifest["input_metadata"]["images"], ["filename", "size", "sha256"])
feature_rows = []
for method in config["methods"]:
    for index in range(4):
        with np.load(run_dir / "features" / f"{method}_{index + 1}.npz") as feature:
            feature_rows.append(
                {
                    "method": method,
                    "image": index + 1,
                    "detected": int(feature["detected_count"]),
                    "used": len(feature["points"]),
                }
            )
table(feature_rows, ["method", "image", "detected", "used"])
for method in config["methods"]:
    show_figure(f"{method}_keypoints_3")


## Parameter grid

The refined grid contains 3 detectors × 3 Lowe ratios (0.72/0.75/0.78) × 4 RANSAC thresholds (3/4/5/6 px), for 36 configurations. Each row records one full-pipeline timing sample, its inlier set and fitted reprojection RMSE. Failed configurations retain their error messages.


In [ ]:
table(
    rows,
    [
        "case_id",
        "status",
        "detected_keypoints",
        "used_keypoints",
        "initial_matches",
        "ratio_pass_matches",
        "inliers",
        "inlier_ratio_pct",
        "rmse_px",
        "canvas_width",
        "canvas_height",
        "total_stitching_ms",
        "timing_kind",
        "error",
    ],
)
for method in config["methods"]:
    show_figure(f"{method}_panorama_grid")
    show_figure(f"{method}_sensitivity")


## Baseline and selected configurations

The baseline is Lowe ratio **0.75**, RANSAC **3 px**. The selected configuration of each method is recorded after inspecting the complete panoramas and fixed anchor-coordinate crops. Repeated timing uses one warmup and five independent full-pipeline repetitions, reported as median and IQR. Selection applies to this sequence and grid.

In [ ]:
if selection["status"] == "reviewed":
    table(
        part_c.read_csv(run_dir / "comparison.csv"),
        [
            "method",
            "case_id",
            "comparison_role",
            "inliers",
            "inlier_ratio_pct",
            "rmse_px",
            "benchmark_status",
            "median_ms",
            "iqr_ms",
            "n_samples",
        ],
    )
    for method, choice in selection["choices"].items():
        print(method.upper(), choice["case_id"], choice["reason"])

    show_figure("baseline_selected_panoramas")
    show_figure("selected_crop_comparison")
    show_figure("runtime_comparison")
else:
    print("Selection status: pending")


## Matching and fixed crops

Match figures show a reproducible subset of at most 80 lines. The full correspondence arrays and inlier masks are saved. Crop coordinates refer to the unchanged image-3 coordinate system, translated into each panorama.

In [ ]:
print("Fixed crops:", figures["roi_boxes_anchor_pixels"])
for method in config["methods"]:
    case_id = (
        selection.get("choices", {})
        .get(method, {})
        .get("case_id", part_c._case_id(method, 0.75, 3.0))
    )
    for source_index in (1, 2, 4):
        show_figure(f"{case_id}_{source_index}_to_3_initial")
        show_figure(f"{case_id}_{source_index}_to_3_inlier")
    if show_all_candidate_crops:
        for name in figures["roi_boxes_anchor_pixels"]:
            show_figure(f"{method}_crop_grid_{name}")


## Input-order sensitivity

The correspondence order is permuted with NumPy seeds **42, 123, 2026**. Inlier masks are restored to the original order for comparison. The main result retains its original correspondence order. Corner deviations are measured in image-3 pixels.


In [ ]:
if (run_dir / "robustness.csv").is_file():
    robustness = part_c.read_csv(run_dir / "robustness.csv")
    table(
        robustness,
        [
            "method",
            "permutation_seed",
            "source_image",
            "status",
            "pipeline_status",
            "inliers",
            "inlier_ratio_pct",
            "rmse_px",
            "corner_rmse_px",
            "corner_max_deviation_px",
            "error",
        ],
    )
    rates = []
    for method in config["methods"]:
        successful = [
            seed
            for seed in (42, 123, 2026)
            if len(
                [
                    row
                    for row in robustness
                    if row["method"] == method and int(row["permutation_seed"]) == seed
                ]
            )
            == 3
            and all(
                row["pipeline_status"] == "ok"
                for row in robustness
                if row["method"] == method and int(row["permutation_seed"]) == seed
            )
        ]
        rates.append(
            {"method": method, "successful_permutations": len(successful), "tested_permutations": 3}
        )
    table(rates, ["method", "successful_permutations", "tested_permutations"])
else:
    print("Order-sensitivity table: not generated for this run.")


if (run_dir / "sensitivity_search_summary.csv").is_file():
    print("Input-order sensitivity across every saved configuration:")
    table(
        part_c.read_csv(run_dir / "sensitivity_search_summary.csv"),
        [
            "case_id",
            "successful_pairs",
            "expected_pairs",
            "max_corner_deviation_px",
            "successful_pipelines",
            "main_rmse_px",
            "main_inliers",
        ],
    )


## Blending and panorama analysis

The detector comparison uses feather blending. This section compares feather with OpenCV multi-band blending at **3, 5 and 7 bands**, holding the selected homographies, canvas and geometric support fixed. Timing covers canvas setup, warp, blend and crop. `mean_abs_change_from_feather` records the mean absolute pixel difference from the feather output.

The tables and crops below display the saved blending comparisons and panorama observations.


In [ ]:
if "input_order_sensitivity" in figures["figures"]:
    show_figure("input_order_sensitivity")
elif (run_dir / "sensitivity_corner_comparison.png").is_file():
    show_image(run_dir / "sensitivity_corner_comparison.png")

if (run_dir / "blending.csv").is_file():
    table(
        part_c.read_csv(run_dir / "blending.csv"),
        [
            "method",
            "case_id",
            "blend",
            "median_ms",
            "iqr_ms",
            "n_samples",
            "mean_abs_change_from_feather",
            "scope",
        ],
    )
    for name in ("panorama", "sign", "column", "lantern", "overlap_detail"):
        show_image(run_dir / "blending" / f"comparison_{name}.png")
else:
    print("Blending table: not generated for this run.")

if (run_dir / "analysis.json").is_file():
    observations = json.loads((run_dir / "analysis.json").read_text())
    table(
        [
            {"aspect": key, "observation": text}
            for key, text in observations.get("qualitative", {}).items()
        ],
        ["aspect", "observation"],
    )
    parameter_notes = []
    for method, notes in observations.get("parameter_sensitivity", {}).items():
        parameter_notes.append(
            {
                "method": method.upper(),
                "ratio_effect": notes.get("observed_ratio_change", ""),
                "threshold_effect": notes.get("observed_threshold_change", ""),
                "selected_case": notes.get("selected_case", ""),
                "max_corner_deviation_px": notes.get("max_corner_deviation_px", ""),
            }
        )
    table(
        parameter_notes,
        ["method", "ratio_effect", "threshold_effect", "selected_case", "max_corner_deviation_px"],
    )
    table(
        [
            {"criterion": key, "finding": text}
            for key, text in observations.get("overall_choice", {}).items()
        ],
        ["criterion", "finding"],
    )

validation = part_c.validate_run(run_dir)
assert validation["valid"], validation["errors"]
print(validation)


## Output conventions and references

Inlier RMSE is the forward reprojection error of matched features after RANSAC. Panorama observations cover alignment, seams, ghosting, illumination and perspective. Exported plots use 300 DPI; notebook previews use PNG.

References: [OpenCV MultiBandBlender](https://docs.opencv.org/4.13.0/d5/d4b/classcv_1_1detail_1_1MultiBandBlender.html), [OpenCV AKAZE](https://docs.opencv.org/5.0/tutorials/features/akaze_matching/akaze_matching.html), [OpenCV RANSAC implementation](https://github.com/opencv/opencv/blob/5.0.0/modules/geometry/src/ptsetreg.cpp).
